# TMVE Cell ID Tracker - Compilador APK

Ejecuta las celdas EN ORDEN, una por una. Al final se descarga la APK.

IMPORTANTE: primero sube `cellid_db.json` a tu repositorio de GitHub antes de ejecutar el Paso 2.

In [ ]:
# PASO 1: Instalar dependencias (~3 min)
%%capture cap
!sudo apt-get update -qq
!sudo apt-get install -y -qq git zip unzip openjdk-17-jdk autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev cmake libffi-dev libssl-dev
!pip install -q buildozer cython==3.0.10
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
print('Listo: dependencias instaladas')


In [ ]:
# PASO 2: Descargar base de datos de Cell ID desde GitHub
import os, urllib.request
os.makedirs('/content/tmve', exist_ok=True)
url = 'https://raw.githubusercontent.com/omendible-hue/tmve-tracker/main/cellid_db.json'
urllib.request.urlretrieve(url, '/content/tmve/cellid_db.json')
import json
with open('/content/tmve/cellid_db.json') as f:
    db = json.load(f)
print(f'Listo: {len(db)} celdas descargadas')


In [ ]:
# PASO 3: Crear main.py
import base64
main_b64 = (
    'aW1wb3J0IGpzb24sIG9zCmZyb20gZGF0ZXRpbWUgaW1wb3J0IGRhdGV0aW1lCmZyb20ga2l2eS5hcHAgaW1wb3J0IEFwcApmcm9tIGtpdnkudWl4LmJveGxheW91dCBpbXBvcnQgQm94TGF5b3V0CmZyb20ga2l2eS51aXgubGFiZWwgaW1wb3J0IExhYmVsCmZyb20g' +
    'a2l2eS51aXguYnV0dG9uIGltcG9ydCBCdXR0b24KZnJvbSBraXZ5LnVpeC5zY3JvbGx2aWV3IGltcG9ydCBTY3JvbGxWaWV3CmZyb20ga2l2eS5jbG9jayBpbXBvcnQgQ2xvY2sKZnJvbSBraXZ5LnV0aWxzIGltcG9ydCBwbGF0Zm9ybQoKaWYgcGxhdGZvcm0gPT0g' +
    'J2FuZHJvaWQnOgogICAgZnJvbSBhbmRyb2lkLnBlcm1pc3Npb25zIGltcG9ydCByZXF1ZXN0X3Blcm1pc3Npb25zLCBQZXJtaXNzaW9uCiAgICBmcm9tIGpuaXVzIGltcG9ydCBhdXRvY2xhc3MKCiAgICBDb250ZXh0ID0gYXV0b2NsYXNzKCdhbmRyb2lkLmNvbnRl' +
    'bnQuQ29udGV4dCcpCiAgICBQeXRob25BY3Rpdml0eSA9IGF1dG9jbGFzcygnb3JnLmtpdnkuYW5kcm9pZC5QeXRob25BY3Rpdml0eScpCiAgICBUZWxlcGhvbnlNYW5hZ2VyID0gYXV0b2NsYXNzKCdhbmRyb2lkLnRlbGVwaG9ueS5UZWxlcGhvbnlNYW5hZ2VyJykK' +
    'ICAgIEJ1aWxkID0gYXV0b2NsYXNzKCdhbmRyb2lkLm9zLkJ1aWxkJykKICAgIFRleHRUb1NwZWVjaCA9IGF1dG9jbGFzcygnYW5kcm9pZC5zcGVlY2gudHRzLlRleHRUb1NwZWVjaCcpCiAgICBMb2NhbGUgPSBhdXRvY2xhc3MoJ2phdmEudXRpbC5Mb2NhbGUnKQoK' +
    'Q0VMTElEX0RCID0ge30gICMga2V5OiBmInt0ZWNofV97Y2VsbGlkfSIgLT4gW25hbWUsIHZlbmRvciwgbWFya2V0XQoKZGVmIGxvYWRfZGIoKToKICAgIGdsb2JhbCBDRUxMSURfREIKICAgIGJhc2UgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9f' +
    'ZmlsZV9fKSkKICAgIHdpdGggb3Blbihvcy5wYXRoLmpvaW4oYmFzZSwgJ2NlbGxpZF9kYi5qc29uJyksICdyJywgZW5jb2Rpbmc9J3V0Zi04JykgYXMgZjoKICAgICAgICByb3dzID0ganNvbi5sb2FkKGYpCiAgICBmb3IgdGVjaCwgY2VsbGlkLCBuYW1lLCB2ZW5k' +
    'b3IsIG1hcmtldCBpbiByb3dzOgogICAgICAgIENFTExJRF9EQltmInt0ZWNofV97Y2VsbGlkfSJdID0gW25hbWUsIHZlbmRvciwgbWFya2V0XQoKCmNsYXNzIENlbGxJZEFwcChBcHApOgogICAgZGVmIGJ1aWxkKHNlbGYpOgogICAgICAgIGxvYWRfZGIoKQogICAg' +
    'ICAgIHNlbGYuYWN0aXZlID0gRmFsc2UKICAgICAgICBzZWxmLmxvZ19saW5lcyA9IFtdCiAgICAgICAgc2VsZi50ZWxlcGhvbnkgPSBOb25lCiAgICAgICAgc2VsZi50dHMgPSBOb25lCiAgICAgICAgc2VsZi5sYXN0X2tleSA9IE5vbmUKCiAgICAgICAgaWYgcGxh' +
    'dGZvcm0gPT0gJ2FuZHJvaWQnOgogICAgICAgICAgICByZXF1ZXN0X3Blcm1pc3Npb25zKFsKICAgICAgICAgICAgICAgIFBlcm1pc3Npb24uUkVBRF9QSE9ORV9TVEFURSwKICAgICAgICAgICAgICAgIFBlcm1pc3Npb24uQUNDRVNTX0ZJTkVfTE9DQVRJT04sCiAg' +
    'ICAgICAgICAgICAgICBQZXJtaXNzaW9uLkFDQ0VTU19DT0FSU0VfTE9DQVRJT04sCiAgICAgICAgICAgICAgICBQZXJtaXNzaW9uLldSSVRFX0VYVEVSTkFMX1NUT1JBR0UsCiAgICAgICAgICAgIF0pCiAgICAgICAgICAgIGFjdGl2aXR5ID0gUHl0aG9uQWN0aXZp' +
    'dHkubUFjdGl2aXR5CiAgICAgICAgICAgIHNlbGYudGVsZXBob255ID0gYWN0aXZpdHkuZ2V0U3lzdGVtU2VydmljZShDb250ZXh0LlRFTEVQSE9OWV9TRVJWSUNFKQogICAgICAgICAgICBzZWxmLnR0cyA9IFRleHRUb1NwZWVjaChhY3Rpdml0eSwgTm9uZSkKICAg' +
    'ICAgICAgICAgc2VsZi50dHMuc2V0TGFuZ3VhZ2UoTG9jYWxlKCdlcycsICdFUycsICcnKSkKCiAgICAgICAgcm9vdCA9IEJveExheW91dChvcmllbnRhdGlvbj0ndmVydGljYWwnLCBwYWRkaW5nPTE0LCBzcGFjaW5nPTgpCiAgICAgICAgcm9vdC5hZGRfd2lkZ2V0' +
    'KExhYmVsKHRleHQ9J1RNVkUgQ2VsbCBJRCBUcmFja2VyJywgZm9udF9zaXplPScyMHNwJywKICAgICAgICAgICAgYm9sZD1UcnVlLCBjb2xvcj0oMC4xLDAuNCwwLjksMSksIHNpemVfaGludF95PU5vbmUsIGhlaWdodD00MCkpCgogICAgICAgIHNlbGYubGJsX3N0' +
    'YXR1cyA9IExhYmVsKHRleHQ9Zid7bGVuKENFTExJRF9EQil9IGNlbGRhcyBlbiBiYXNlIGRlIGRhdG9zJywKICAgICAgICAgICAgZm9udF9zaXplPScxM3NwJywgc2l6ZV9oaW50X3k9Tm9uZSwgaGVpZ2h0PTI2KQogICAgICAgIHJvb3QuYWRkX3dpZGdldChzZWxm' +
    'LmxibF9zdGF0dXMpCgogICAgICAgIHNlbGYubGJsX3RlY2ggPSBMYWJlbCh0ZXh0PSdSZWQ6IC0tLSAgfCAgQ2VsbCBJRDogLS0tJywgZm9udF9zaXplPScxNXNwJywKICAgICAgICAgICAgc2l6ZV9oaW50X3k9Tm9uZSwgaGVpZ2h0PTMwKQogICAgICAgIHJvb3Qu' +
    'YWRkX3dpZGdldChzZWxmLmxibF90ZWNoKQoKICAgICAgICBzZWxmLmxibF9zdGF0aW9uID0gTGFiZWwodGV4dD0nRXN0YWNpb246IC0tLScsIGZvbnRfc2l6ZT0nMjBzcCcsCiAgICAgICAgICAgIGJvbGQ9VHJ1ZSwgY29sb3I9KDAuMDUsMC41LDAuMSwxKSwgc2l6' +
    'ZV9oaW50X3k9Tm9uZSwgaGVpZ2h0PTQ0KQogICAgICAgIHJvb3QuYWRkX3dpZGdldChzZWxmLmxibF9zdGF0aW9uKQoKICAgICAgICBzZWxmLmxibF9leHRyYSA9IExhYmVsKHRleHQ9J1ZlbmRvcjogLS0tICB8ICBNZXJjYWRvOiAtLS0nLAogICAgICAgICAgICBm' +
    'b250X3NpemU9JzEzc3AnLCBzaXplX2hpbnRfeT1Ob25lLCBoZWlnaHQ9MjYpCiAgICAgICAgcm9vdC5hZGRfd2lkZ2V0KHNlbGYubGJsX2V4dHJhKQoKICAgICAgICBidG5fcm93ID0gQm94TGF5b3V0KHNpemVfaGludF95PU5vbmUsIGhlaWdodD00OCwgc3BhY2lu' +
    'Zz0xMCkKICAgICAgICBzZWxmLmJ0bl9zdGFydCA9IEJ1dHRvbih0ZXh0PSdJbmljaWFyJywgYmFja2dyb3VuZF9jb2xvcj0oMC4xLDAuNCwwLjgsMSkpCiAgICAgICAgc2VsZi5idG5fc3RhcnQuYmluZChvbl9wcmVzcz1zZWxmLnN0YXJ0X3RyYWNraW5nKQogICAg' +
    'ICAgIHNlbGYuYnRuX3N0b3AgPSBCdXR0b24odGV4dD0nRGV0ZW5lcicsIGJhY2tncm91bmRfY29sb3I9KDAuNywwLjEsMC4xLDEpLCBkaXNhYmxlZD1UcnVlKQogICAgICAgIHNlbGYuYnRuX3N0b3AuYmluZChvbl9wcmVzcz1zZWxmLnN0b3BfdHJhY2tpbmcpCiAg' +
    'ICAgICAgYnRuX3Jvdy5hZGRfd2lkZ2V0KHNlbGYuYnRuX3N0YXJ0KQogICAgICAgIGJ0bl9yb3cuYWRkX3dpZGdldChzZWxmLmJ0bl9zdG9wKQogICAgICAgIHJvb3QuYWRkX3dpZGdldChidG5fcm93KQoKICAgICAgICByb290LmFkZF93aWRnZXQoTGFiZWwodGV4' +
    'dD0nSGlzdG9yaWFsOicsIGZvbnRfc2l6ZT0nMTRzcCcsIGJvbGQ9VHJ1ZSwKICAgICAgICAgICAgc2l6ZV9oaW50X3k9Tm9uZSwgaGVpZ2h0PTI0KSkKICAgICAgICBzdiA9IFNjcm9sbFZpZXcoKQogICAgICAgIHNlbGYubGJsX2xvZyA9IExhYmVsKHRleHQ9Jycs' +
    'IGZvbnRfc2l6ZT0nMTJzcCcsIHNpemVfaGludF95PU5vbmUsCiAgICAgICAgICAgIGhhbGlnbj0nbGVmdCcsIHZhbGlnbj0ndG9wJykKICAgICAgICBzZWxmLmxibF9sb2cuYmluZCh0ZXh0dXJlX3NpemU9c2VsZi5sYmxfbG9nLnNldHRlcignc2l6ZScpKQogICAg' +
    'ICAgIHN2LmFkZF93aWRnZXQoc2VsZi5sYmxfbG9nKQogICAgICAgIHJvb3QuYWRkX3dpZGdldChzdikKCiAgICAgICAgYnRuX2V4cCA9IEJ1dHRvbih0ZXh0PSdFeHBvcnRhciBsb2cnLCBzaXplX2hpbnRfeT1Ob25lLCBoZWlnaHQ9NDQpCiAgICAgICAgYnRuX2V4' +
    'cC5iaW5kKG9uX3ByZXNzPXNlbGYuZXhwb3J0X2xvZykKICAgICAgICByb290LmFkZF93aWRnZXQoYnRuX2V4cCkKCiAgICAgICAgcmV0dXJuIHJvb3QKCiAgICBkZWYgZ2V0X25ldHdvcmtfdGVjaChzZWxmLCBuZXRfdHlwZSk6CiAgICAgICAgIyBBbmRyb2lkIFRl' +
    'bGVwaG9ueU1hbmFnZXIgTkVUV09SS19UWVBFIGNvbnN0YW50cwogICAgICAgIGdzbSA9ICgxLCAyLCAxNikgICAgICAgICAgICAgICAjIEdQUlMsIEVER0UsIEdTTQogICAgICAgIHVtdHMgPSAoMywgOCwgOSwgMTAsIDE1LCAxNykgICAjIFVNVFMsIEhTRFBBLCBI' +
    'U1VQQSwgSFNQQSwgSFNQQSsKICAgICAgICBsdGUgPSAoMTMsIDE5KSAgICAgICAgICAgICAgICAgICMgTFRFLCBMVEVfQ0EKICAgICAgICBuciA9ICgyMCwpICAgICAgICAgICAgICAgICAgICAgICMgTlIgKDVHKQogICAgICAgIGlmIG5ldF90eXBlIGluIGdzbTog' +
    'cmV0dXJuICcyRycKICAgICAgICBpZiBuZXRfdHlwZSBpbiB1bXRzOiByZXR1cm4gJzNHJwogICAgICAgIGlmIG5ldF90eXBlIGluIGx0ZTogcmV0dXJuICc0RycKICAgICAgICBpZiBuZXRfdHlwZSBpbiBucjogcmV0dXJuICc1RycKICAgICAgICByZXR1cm4gZic/' +
    'KHtuZXRfdHlwZX0pJwoKICAgIGRlZiByZWFkX2NlbGxfaW5mbyhzZWxmKToKICAgICAgICBpZiBwbGF0Zm9ybSAhPSAnYW5kcm9pZCcgb3Igbm90IHNlbGYudGVsZXBob255OgogICAgICAgICAgICBpbXBvcnQgcmFuZG9tCiAgICAgICAgICAgIHRlY2ggPSByYW5k' +
    'b20uY2hvaWNlKFsnMkcnLCAnM0cnLCAnNEcnXSkKICAgICAgICAgICAgY2VsbGlkID0gcmFuZG9tLmNob2ljZShsaXN0KENFTExJRF9EQi5rZXlzKCkpKQogICAgICAgICAgICB0LCBjaWQgPSBjZWxsaWQuc3BsaXQoJ18nLCAxKQogICAgICAgICAgICByZXR1cm4g' +
    'dCwgY2lkCgogICAgICAgIHRyeToKICAgICAgICAgICAgbmV0X3R5cGUgPSBzZWxmLnRlbGVwaG9ueS5nZXROZXR3b3JrVHlwZSgpCiAgICAgICAgICAgIHRlY2ggPSBzZWxmLmdldF9uZXR3b3JrX3RlY2gobmV0X3R5cGUpCiAgICAgICAgICAgIGNlbGxpZCA9ICct' +
    'LS0nCgogICAgICAgICAgICBjZWxsX2xvY2F0aW9uID0gc2VsZi50ZWxlcGhvbnkuZ2V0Q2VsbExvY2F0aW9uKCkKICAgICAgICAgICAgaWYgY2VsbF9sb2NhdGlvbjoKICAgICAgICAgICAgICAgIGNpZF9tZXRob2QgPSBnZXRhdHRyKGNlbGxfbG9jYXRpb24sICdn' +
    'ZXRDaWQnLCBOb25lKQogICAgICAgICAgICAgICAgaWYgY2lkX21ldGhvZDoKICAgICAgICAgICAgICAgICAgICByYXdfY2lkID0gY2lkX21ldGhvZCgpCiAgICAgICAgICAgICAgICAgICAgaWYgcmF3X2NpZCBhbmQgcmF3X2NpZCA+IDA6CiAgICAgICAgICAgICAg' +
    'ICAgICAgICAgIGNlbGxpZCA9IHN0cihyYXdfY2lkKQoKICAgICAgICAgICAgcmV0dXJuIHRlY2gsIGNlbGxpZAogICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZToKICAgICAgICAgICAgcmV0dXJuIGYnRXJyb3InLCBzdHIoZSkKCiAgICBkZWYgdXBkYXRlX2Rp' +
    'c3BsYXkoc2VsZiwgZHQpOgogICAgICAgIGlmIG5vdCBzZWxmLmFjdGl2ZToKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgdGVjaCwgY2VsbGlkID0gc2VsZi5yZWFkX2NlbGxfaW5mbygpCiAgICAgICAgc2VsZi5sYmxfdGVjaC50ZXh0ID0gZidSZWQ6IHt0ZWNo' +
    'fSAgfCAgQ2VsbCBJRDoge2NlbGxpZH0nCgogICAgICAgIGtleSA9IGYie3RlY2h9X3tjZWxsaWR9IgogICAgICAgIGluZm8gPSBDRUxMSURfREIuZ2V0KGtleSkKCiAgICAgICAgaWYgaW5mbzoKICAgICAgICAgICAgbmFtZSwgdmVuZG9yLCBtYXJrZXQgPSBpbmZv' +
    'CiAgICAgICAgICAgIHNlbGYubGJsX3N0YXRpb24udGV4dCA9IGYnRXN0YWNpb246IHtuYW1lfScKICAgICAgICAgICAgc2VsZi5sYmxfZXh0cmEudGV4dCA9IGYnVmVuZG9yOiB7dmVuZG9yfSAgfCAgTWVyY2Fkbzoge21hcmtldH0nCiAgICAgICAgZWxzZToKICAg' +
    'ICAgICAgICAgc2VsZi5sYmxfc3RhdGlvbi50ZXh0ID0gZidFc3RhY2lvbjogTm8gZW5jb250cmFkYSAoQ2VsbCBJRCB7Y2VsbGlkfSknCiAgICAgICAgICAgIHNlbGYubGJsX2V4dHJhLnRleHQgPSBmJ1ZlbmRvcjogLS0tICB8ICBNZXJjYWRvOiAtLS0nCgogICAg' +
    'ICAgIGlmIGtleSAhPSBzZWxmLmxhc3Rfa2V5OgogICAgICAgICAgICBzZWxmLmxhc3Rfa2V5ID0ga2V5CiAgICAgICAgICAgIGhvcmEgPSBkYXRldGltZS5ub3coKS5zdHJmdGltZSgnJUg6JU06JVMnKQogICAgICAgICAgICBub21icmUgPSBpbmZvWzBdIGlmIGlu' +
    'Zm8gZWxzZSBmJ0Rlc2Nvbm9jaWRhICh7Y2VsbGlkfSknCiAgICAgICAgICAgIGxpbmUgPSBmIntob3JhfSBbe3RlY2h9XSBDZWxsSUQ6e2NlbGxpZH0gLT4ge25vbWJyZX0iCiAgICAgICAgICAgIHNlbGYubG9nX2xpbmVzLmFwcGVuZChsaW5lKQogICAgICAgICAg' +
    'ICBzZWxmLmxibF9sb2cudGV4dCA9IGNocigxMCkuam9pbihzZWxmLmxvZ19saW5lc1stNTA6XSkKICAgICAgICAgICAgaWYgc2VsZi50dHMgYW5kIGluZm86CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgc2VsZi50dHMuc3BlYWsoZidF' +
    'c3RhY2lvbjoge2luZm9bMF19JywgVGV4dFRvU3BlZWNoLlFVRVVFX0ZMVVNILCBOb25lLCBOb25lKQogICAgICAgICAgICAgICAgZXhjZXB0OgogICAgICAgICAgICAgICAgICAgIHBhc3MKCiAgICBkZWYgc3RhcnRfdHJhY2tpbmcoc2VsZiwgKmEpOgogICAgICAg' +
    'IHNlbGYuYWN0aXZlID0gVHJ1ZQogICAgICAgIHNlbGYuYnRuX3N0YXJ0LmRpc2FibGVkID0gVHJ1ZQogICAgICAgIHNlbGYuYnRuX3N0b3AuZGlzYWJsZWQgPSBGYWxzZQogICAgICAgIHNlbGYubGJsX3N0YXR1cy50ZXh0ID0gZid7bGVuKENFTExJRF9EQil9IGNl' +
    'bGRhcyAtIExleWVuZG8gY2FkYSA1cy4uLicKICAgICAgICBDbG9jay5zY2hlZHVsZV9pbnRlcnZhbChzZWxmLnVwZGF0ZV9kaXNwbGF5LCA1KQoKICAgIGRlZiBzdG9wX3RyYWNraW5nKHNlbGYsICphKToKICAgICAgICBzZWxmLmFjdGl2ZSA9IEZhbHNlCiAgICAg' +
    'ICAgc2VsZi5idG5fc3RhcnQuZGlzYWJsZWQgPSBGYWxzZQogICAgICAgIHNlbGYuYnRuX3N0b3AuZGlzYWJsZWQgPSBUcnVlCiAgICAgICAgc2VsZi5sYmxfc3RhdHVzLnRleHQgPSBmJ3tsZW4oQ0VMTElEX0RCKX0gY2VsZGFzIC0gRGV0ZW5pZG8nCiAgICAgICAg' +
    'Q2xvY2sudW5zY2hlZHVsZShzZWxmLnVwZGF0ZV9kaXNwbGF5KQoKICAgIGRlZiBleHBvcnRfbG9nKHNlbGYsICphKToKICAgICAgICBwYXRoID0gJy9zZGNhcmQvY2VsbGlkX2xvZy50eHQnIGlmIHBsYXRmb3JtID09ICdhbmRyb2lkJyBlbHNlICdjZWxsaWRfbG9n' +
    'LnR4dCcKICAgICAgICB3aXRoIG9wZW4ocGF0aCwgJ3cnKSBhcyBmOgogICAgICAgICAgICBmLndyaXRlKGNocigxMCkuam9pbihzZWxmLmxvZ19saW5lcykpCiAgICAgICAgc2VsZi5sYmxfc3RhdHVzLnRleHQgPSBmJ0d1YXJkYWRvOiB7cGF0aH0nCgoKaWYgX19u' +
    'YW1lX18gPT0gJ19fbWFpbl9fJzoKICAgIENlbGxJZEFwcCgpLnJ1bigpCg=='
)
with open('/content/tmve/main.py', 'wb') as f:
    f.write(base64.b64decode(main_b64))
print('Listo: main.py creado')


In [ ]:
# PASO 4: Crear buildozer.spec
import base64
spec_b64 = (
    'W2FwcF0KdGl0bGUgPSBUTVZFIENlbGxJRApwYWNrYWdlLm5hbWUgPSB0bXZlY2VsbGlkCnBhY2thZ2UuZG9tYWluID0gb3JnLnRtdmUKc291cmNlLmRpciA9IC4Kc291cmNlLmluY2x1ZGVfZXh0cyA9IHB5LGpzb24KdmVyc2lvbiA9IDEuMApyZXF1aXJlbWVudHMg' +
    'PSBweXRob24zLGtpdnkscHlqbml1cwpvcmllbnRhdGlvbiA9IHBvcnRyYWl0CmFuZHJvaWQucGVybWlzc2lvbnMgPSBSRUFEX1BIT05FX1NUQVRFLEFDQ0VTU19GSU5FX0xPQ0FUSU9OLEFDQ0VTU19DT0FSU0VfTE9DQVRJT04sV1JJVEVfRVhURVJOQUxfU1RPUkFH' +
    'RQphbmRyb2lkLmFwaSA9IDMxCmFuZHJvaWQubWluYXBpID0gMjEKYW5kcm9pZC5uZGsgPSAyNWIKYW5kcm9pZC5zZGsgPSAzMQphbmRyb2lkLmJ1aWxkX3Rvb2xzX3ZlcnNpb24gPSAzMC4wLjMKYW5kcm9pZC5hcmNocyA9IGFybTY0LXY4YQpmdWxsc2NyZWVuID0g' +
    'MAoKW2J1aWxkb3plcl0KbG9nX2xldmVsID0gMgp3YXJuX29uX3Jvb3QgPSAxCg=='
)
with open('/content/tmve/buildozer.spec', 'wb') as f:
    f.write(base64.b64decode(spec_b64))
print('Listo: buildozer.spec creado')


In [ ]:
# PASO 5: Compilar APK (~20-25 minutos, no cierres la pestana)
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
os.chdir('/content/tmve')
!mkdir -p ~/.android
!echo 'count=0' > ~/.android/repositories.cfg
print('Compilando, esto tarda unos 20-25 minutos...')
!buildozer android debug 2>&1


In [ ]:
# PASO 6: Descargar APK
import glob
from google.colab import files
apks = glob.glob('/content/tmve/bin/*.apk')
if apks:
    print(f'APK lista: {apks[0]}')
    files.download(apks[0])
else:
    print('ERROR: no se genero APK, revisa el log del paso 5')
